# torchlight: fine-tune Phi-4-mini on an NVIDIA GPU with Unsloth

This notebook fine-tunes Phi-4-mini with [Unsloth](https://github.com/unslothai/unsloth) on a CUDA GPU, with the same data, prompt and hyperparameters as `torchlight train` on a Mac. It follows Unsloth's defaults of a 4-bit base and LoRA on every layer, where the Mac run uses the 16-bit base and the top 16 layers to save time.

- data: the `sft` config of [`samadpls/torchlight-threads`](https://huggingface.co/datasets/samadpls/torchlight-threads), with the system prompt torchlight trains with;
- loss on the final assistant message of each chat only, the answer that resolved the thread;
- LoRA rank 16, alpha 32, on a 4-bit base; learning rate 2e-4 with cosine decay after 5% warmup; 8 chats per update; one epoch;
- output: the LoRA adapter and the adapter merged into the 16-bit model.

Runtime: Colab (Runtime > Change runtime type > GPU) or Kaggle (Accelerator > GPU). An A100 or L4 finishes in well under an hour, a T4 in a few hours.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
BASE_MODEL = "unsloth/Phi-4-mini-instruct"
DATASET = "samadpls/torchlight-threads"
MAX_LENGTH = 4096
OUTPUT_DIR = "torchlight-phi4-mini"
SYSTEM_PROMPT = 'You are a careful PyTorch helper. Give specific, correct guidance with code where it helps. If a question lacks the details needed to answer it, ask for them, for example a minimal executable code snippet or the full error message, instead of guessing.'

# Set to a repo id such as "your-name/torchlight-phi4-mini" to upload the merged model.
HUB_REPO = None

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL, max_seq_length=MAX_LENGTH, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    # Phi-3 style fused projections, and the split names in case the checkpoint uses them.
    target_modules=[
        "qkv_proj", "gate_up_proj", "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=0,
)

Each row's `messages` ends with the assistant reply that resolved the thread. Earlier assistant turns can be suggestions that didn't work, so only the final message is trained: every token before it gets the label `-100`. Chats longer than `MAX_LENGTH` are dropped rather than truncated, since truncation would cut the answer.

In [ ]:
from datasets import load_dataset


def tokenize(row):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + row["messages"]
    prompt = tokenizer.apply_chat_template(
        messages[:-1], add_generation_prompt=True, tokenize=True, return_dict=False
    )
    full = tokenizer.apply_chat_template(messages, tokenize=True, return_dict=False)
    return {
        "input_ids": full,
        "attention_mask": [1] * len(full),
        "labels": [-100] * len(prompt) + full[len(prompt):],
    }


sft = load_dataset(DATASET, "sft")
columns = sft["train"].column_names
train = sft["train"].map(tokenize, remove_columns=columns)
valid = sft["validation"].map(tokenize, remove_columns=columns)
train = train.filter(lambda row: len(row["input_ids"]) <= MAX_LENGTH)
valid = valid.filter(lambda row: len(row["input_ids"]) <= MAX_LENGTH)
print(f"train: {len(train)} chats, validation: {len(valid)} chats")

In [ ]:
import torch
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

bf16 = torch.cuda.is_bf16_supported()
trainer = Trainer(
    model=model,
    train_dataset=train,
    eval_dataset=valid,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        optim="adamw_8bit",
        bf16=bf16,
        fp16=not bf16,
        group_by_length=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        seed=0,
        report_to="none",
    ),
)
trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)
question = "My loss becomes NaN after a few iterations when I use mixed precision. What can I check?"
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to("cuda")
output = model.generate(**inputs, max_new_tokens=400, do_sample=False)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")
model.save_pretrained_merged(f"{OUTPUT_DIR}/merged", tokenizer, save_method="merged_16bit")

if HUB_REPO:
    model.push_to_hub_merged(HUB_REPO, tokenizer, save_method="merged_16bit")